## Export S2-only model

In [ ]:
!uv pip install onnx onnxscript requests aiohttp segmentation_models_pytorch fsspec

In [ ]:
import io

import fsspec
import segmentation_models_pytorch as smp
import torch
from torch.export import Dim

EPS = 1e-10


class SKeMaModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        in_channels = 10
        self.model = smp.Unet(
            encoder_name="tu-maxvit_tiny_tf_512",
            in_channels=in_channels,
            encoder_weights=None,
        )

        self.register_buffer(
            "per_channel_mean",
            torch.zeros((1, in_channels, 1, 1)),
        )

        self.register_buffer(
            "per_channel_std",
            torch.ones((1, in_channels, 1, 1)),
        )

    def forward(self, x):
        # Unpack spectral bands
        blue = x.select(1, 0).unsqueeze(1)
        green = x.select(1, 1).unsqueeze(1)
        red = x.select(1, 2).unsqueeze(1)
        nir = x.select(1, 3).unsqueeze(1)
        re = x.select(1, 4).unsqueeze(1)

        # Compute vegetation indices
        ndvi = self.normalized_index(nir, red)
        gndvi = self.normalized_index(nir, green)
        ndvi_re = self.normalized_index(re, red)

        # Compute other indices
        ndwi = self.normalized_index(green, nir)
        chl_green = (nir / (green + EPS)) - 1  # Chlorophyll Index Green

        # Stack all bands and indices
        x_aug = torch.cat([blue, green, red, nir, re, ndvi, ndwi, gndvi, chl_green, ndvi_re], dim=1)

        x_aug_normalized = (x_aug - self.per_channel_mean) / self.per_channel_std

        return self.model(x_aug_normalized)

    @staticmethod
    def normalized_index(a, b):
        return (a - b) / (a + b + EPS)


model = SKeMaModel()

sample_input = torch.rand((2, 5, 512, 512), device=torch.device("cpu"), requires_grad=False)
model(sample_input)

In [ ]:
with fsspec.open("https://huggingface.co/m5ghanba/SKeMa/resolve/main/modelS2Only.pth") as f:
    bytes_io = io.BytesIO(f.read())
state_dict = torch.load(bytes_io, map_location="cpu")

In [ ]:
# Update keys
del state_dict["mean"]
state_dict["per_channel_mean"] = torch.tensor([
    2.08900522e02,
    2.70272557e02,
    1.52312422e02,
    9.87182507e02,
    3.26650321e02,
    5.65647882e-02,
    1.62913280e-01,
    -1.62913280e-01,
    1.90832012e07,
    1.09887867e-01,
]).view(1, -1, 1, 1)

del state_dict["std"]
state_dict["per_channel_std"] = torch.tensor([
    1.59021908e02,
    2.18393833e02,
    2.08355086e02,
    1.29667310e03,
    3.79794112e02,
    6.53314129e-01,
    7.11820223e-01,
    7.11820223e-01,
    2.07060299e10,
    3.90619966e-01,
]).view(1, -1, 1, 1)

model.load_state_dict(state_dict, strict=True)
model.eval()

In [ ]:
outpath = "./Unet_tu-maxvit_tiny_tf_512_20260222.onnx"

img_shape = (Dim("batch"), Dim.STATIC, Dim.STATIC, Dim.STATIC)
args = (sample_input,)

with torch.no_grad():
    torch.onnx.export(
        model,
        args,
        outpath,
        # opset_version=22
        export_params=True,
        do_constant_folding=True,
        input_names=["input"],
        output_names=["output"],
        # dynamic_shapes=(img_shape,),
        dynamic_axes={
            "input": {0: "batch_size"},
            "output": {0: "batch_size"},
        },
        verbose=True,
        dynamo=False,
    )

### Model pruning

In [ ]:
import pandas as pd
import torch.nn.utils.prune as prune

# stem.conv1 weight shape: (64, 10, 3, 3)
# For each input channel, sum the L1 norm across all output filters and spatial dims.
# High value = the first layer collectively relies heavily on this input channel.
channel_names = ["Blue", "Green", "Red", "NIR", "RedEdge", "NDVI", "GNDVI", "NDVI_RE", "NDWI", "ChlGreen"]
stem_weight = model.model.encoder.model.stem.conv1.weight  # (64, 10, 3, 3)
input_channel_importance = stem_weight.detach().abs().sum(dim=(0, 2, 3))  # (10,)

importance_df = (
    pd.DataFrame({
        "channel": channel_names,
        "l1_importance": input_channel_importance.numpy(),
        "relative_importance": (input_channel_importance / input_channel_importance.max()).numpy(),
    })
    .sort_values("l1_importance")
    .reset_index(drop=True)
)

print("Input channel importance (ascending):")
print(importance_df.to_string(index=False, float_format="{:.4f}".format))

In [ ]:
import copy

SPARSITY_LEVELS = [0.05, 0.10, 0.20, 0.30]
DEVIATION_TOLERANCE = 0.01  # L-inf deviation as fraction of output range

# MaxViT requires exactly 512x512; use batch=1 to keep memory/time manageable
analysis_input = torch.rand((1, 5, 512, 512), requires_grad=False)

model.eval()
with torch.no_grad():
    baseline_out = model(analysis_input)
output_range = (baseline_out.max() - baseline_out.min()).item()


def get_prunable_conv_layers(module, prefix=""):
    """Recursively collect Conv2d layers with >1 output channel.

    Returns:
        List of (name, layer) tuples.
    """
    layers = []
    for name, child in module.named_children():
        full_name = f"{prefix}.{name}" if prefix else name
        if isinstance(child, torch.nn.Conv2d) and child.out_channels > 1:
            layers.append((full_name, child))
        else:
            layers.extend(get_prunable_conv_layers(child, full_name))
    return layers


conv_layers = get_prunable_conv_layers(model)
print(f"Found {len(conv_layers)} prunable Conv2d layers")

results = []
for layer_name, _ in conv_layers:
    row = {"layer": layer_name}
    for sparsity in SPARSITY_LEVELS:
        test_model = copy.deepcopy(model)
        test_model.eval()
        test_layer = test_model
        for part in layer_name.split("."):
            test_layer = getattr(test_layer, part)

        prune.ln_structured(test_layer, name="weight", amount=sparsity, n=1, dim=0)
        prune.remove(test_layer, "weight")

        with torch.no_grad():
            test_out = test_model(analysis_input)

        deviation = (test_out - baseline_out).abs().max().item() / (output_range + 1e-10)
        row[f"dev@{int(sparsity * 100)}%"] = round(deviation, 4)

    results.append(row)

sensitivity_df = pd.DataFrame(results)


def safe_sparsity(row):
    for s in SPARSITY_LEVELS:
        if row[f"dev@{int(s * 100)}%"] > DEVIATION_TOLERANCE:
            idx = SPARSITY_LEVELS.index(s)
            return SPARSITY_LEVELS[idx - 1] if idx > 0 else 0.0
    return SPARSITY_LEVELS[-1]


sensitivity_df["safe_sparsity"] = sensitivity_df.apply(safe_sparsity, axis=1)

print("\nSensitivity results (layers with safe_sparsity > 0):")
prunable = sensitivity_df[sensitivity_df["safe_sparsity"] > 0].copy()
print(prunable.to_string(index=False, float_format="{:.4f}".format))
print(f"\n{len(prunable)} / {len(sensitivity_df)} layers can be safely pruned")

In [ ]:
params_before = sum(p.numel() for p in model.parameters())

# Apply output-channel structured pruning at the safe sparsity rate for each layer.
# Note: prune.ln_structured zeros out filters but keeps tensor shapes — parameter count
# stays the same. The actual ONNX speedup comes from constant-folding zero-weight paths
# and from the input channel drop below (which genuinely shrinks stem.conv1).
for _, row in sensitivity_df[sensitivity_df["safe_sparsity"] > 0].iterrows():
    layer = model
    for part in row["layer"].split("."):
        layer = getattr(layer, part)
    prune.ln_structured(layer, name="weight", amount=row["safe_sparsity"], n=1, dim=0)
    prune.remove(layer, "weight")

# --- Optional: drop redundant input channels from stem.conv1 ---
# keep_indices / dropped_channels are in the 10-channel augmented space (post-NDVI).
# Dropping a channel here requires also updating forward() to stop computing/concatenating it.
max_importance = input_channel_importance.max().item()
drop_mask = (input_channel_importance / max_importance) < 0.10
dropped_channels = [channel_names[i] for i, d in enumerate(drop_mask.tolist()) if d]
keep_indices = [i for i, d in enumerate(drop_mask.tolist()) if not d]

if dropped_channels:
    print(f"Dropping input channels: {dropped_channels}")

    conv = model.model.encoder.model.stem.conv1
    conv.weight = torch.nn.Parameter(conv.weight[:, keep_indices, :, :])
    conv.in_channels = len(keep_indices)

    # per_channel_mean / per_channel_std are registered buffers, not Parameters
    model.register_buffer("per_channel_mean", model.per_channel_mean[:, keep_indices, :, :])
    model.register_buffer("per_channel_std", model.per_channel_std[:, keep_indices, :, :])

    print("\nManually update SKeMaModel.forward() to remove these from the x_aug cat():")
    for ch in dropped_channels:
        print(f"  - {ch}")
    print("Also remove the computation of any dropped derived index.")
    print("Re-run this cell after updating forward() to verify deviation.")
else:
    print("No input channels dropped (all above 10% relative importance threshold)")

params_after = sum(p.numel() for p in model.parameters())
print(f"\nParameters before: {params_before:,}")
print(f"Parameters after:  {params_after:,}")
print(f"Parameter delta (zeros added): {params_before - params_after:,}")
print("(Shape-preserving pruning keeps count identical; zeros suppress those feature maps at inference)")

# Verify end-to-end output deviation. analysis_input is always 5-channel raw bands —
# the model's forward() handles augmentation internally.
with torch.no_grad():
    pruned_out = model(analysis_input)

deviation = (pruned_out - baseline_out).abs().max().item() / (output_range + 1e-10)
print(f"\nFinal L-inf output deviation: {deviation:.4f} ({deviation * 100:.2f}% of output range)")

In [ ]:
import os

pruned_outpath = "./Unet_tu-maxvit_tiny_tf_512_20260222_pruned.onnx"

# Raw input is always 5-channel; forward() handles augmentation internally.
# If input channels were dropped from stem.conv1, forward() must be updated first
# (to produce fewer augmented channels) before this export will succeed.
pruned_sample = torch.rand((2, 5, 512, 512), requires_grad=False)

with torch.no_grad():
    torch.onnx.export(
        model,
        (pruned_sample,),
        pruned_outpath,
        export_params=True,
        do_constant_folding=True,
        input_names=["input"],
        output_names=["output"],
        dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
        dynamo=False,
    )

original_mb = os.path.getsize(outpath) / 1e6
pruned_mb = os.path.getsize(pruned_outpath) / 1e6
print(f"Original ONNX: {original_mb:.1f} MB")
print(f"Pruned ONNX:   {pruned_mb:.1f} MB")
print(f"Size reduction: {100 * (1 - pruned_mb / original_mb):.1f}%")

### Post-training quantization

In [ ]:
import time

import numpy as np
import onnxruntime as ort
from onnxruntime.quantization import QuantType, quantize_dynamic

quantized_outpath = "./Unet_tu-maxvit_tiny_tf_512_20260222_pruned_int8.onnx"

quantize_dynamic(
    model_input=pruned_outpath,
    model_output=quantized_outpath,
    weight_type=QuantType.QInt8,
    per_channel=True,  # per-channel scales improve accuracy vs global scale
    reduce_range=True,  # avoids overflow on AVX2/AVX-512 CPUs
)

quantized_mb = os.path.getsize(quantized_outpath) / 1e6
print(f"Pruned FP32 ONNX:  {pruned_mb:.1f} MB")
print(f"Quantized INT8:    {quantized_mb:.1f} MB")
print(f"Size reduction:    {100 * (1 - quantized_mb / pruned_mb):.1f}%")

# --- Latency comparison ---
# NOTE: INT8 dynamic quantization is slower on Apple Silicon (no native INT8 SIMD via
# CPUExecutionProvider). On Linux x86 with AVX-512 VNNI or a CUDA device, expect 1.5-3x
# speedup. Use TensorrtExecutionProvider on CUDA for best results.
rng = np.random.default_rng()
timing_input = rng.random((1, 5, 512, 512)).astype(np.float32)

sess_fp32 = ort.InferenceSession(pruned_outpath, providers=["CPUExecutionProvider"])
sess_int8 = ort.InferenceSession(quantized_outpath, providers=["CPUExecutionProvider"])

N_WARMUP, N_RUNS = 2, 5

for sess in (sess_fp32, sess_int8):
    for _ in range(N_WARMUP):
        sess.run(None, {"input": timing_input})

times_fp32, times_int8 = [], []
for _ in range(N_RUNS):
    t0 = time.perf_counter()
    sess_fp32.run(None, {"input": timing_input})
    times_fp32.append(time.perf_counter() - t0)

for _ in range(N_RUNS):
    t0 = time.perf_counter()
    sess_int8.run(None, {"input": timing_input})
    times_int8.append(time.perf_counter() - t0)

fp32_ms = np.median(times_fp32) * 1000
int8_ms = np.median(times_int8) * 1000
print(f"\nMedian latency (1x512x512, CPU, {N_RUNS} runs):")
print(f"  Pruned FP32: {fp32_ms:.0f} ms")
print(f"  INT8:        {int8_ms:.0f} ms  ({fp32_ms / int8_ms:.2f}x speedup)")

## Export S2 + Bathymetry + Substrate model

In [ ]:
EPS = 1e-10


class SKeMaBathyModel(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.model = smp.Unet(
            encoder_name="tu-maxvit_tiny_tf_512",
            in_channels=12,
            encoder_weights=None,
        )

        self.register_buffer(
            "per_channel_mean",
            torch.tensor([
                2.02127847e02,
                2.64991799e02,
                1.45913497e02,
                9.57456953e02,
                3.20302883e02,
                1.37548690e00,
                -3.29322396e-01,
                3.47405153e01,
                3.66107438e-02,
                1.84492036e-01,
                -1.84492036e-01,
                2.22893410e07,
                9.99078992e-02,
            ]).view(1, -1, 1, 1),
        )

        self.register_buffer(
            "per_channel_std",
            torch.tensor([
                1.61504107e02,
                2.22303637e02,
                2.03997451e02,
                1.26105656e03,
                3.79069759e02,
                1.36767747e00,
                2.02345453e02,
                2.60894243e01,
                6.71879776e-01,
                7.23202999e-01,
                7.23202999e-01,
                2.25708723e10,
                4.06695959e-01,
            ]).view(1, -1, 1, 1),
        )

    def forward(self, x):
        # Unpack spectral bands
        blue = x.select(1, 0).unsqueeze(1)
        green = x.select(1, 1).unsqueeze(1)
        red = x.select(1, 2).unsqueeze(1)
        nir = x.select(1, 3).unsqueeze(1)
        re = x.select(1, 4).unsqueeze(1)
        substrate = x.select(1, 5).unsqueeze(1)
        bathymetry = x.select(1, 6).unsqueeze(1)

        # Compute vegetation indices
        ndvi = self.normalized_index(nir, red)
        gndvi = self.normalized_index(nir, green)
        ndvi_re = self.normalized_index(re, red)

        # Compute other indices
        ndwi = self.normalized_index(green, nir)
        chl_green = (nir / (green + EPS)) - 1  # Chlorophyll Index Green

        # Stack all bands and indices
        x_aug = torch.cat(
            [blue, green, red, nir, re, substrate, bathymetry, ndvi, ndwi, gndvi, chl_green, ndvi_re], dim=1
        )

        x_aug_normalized = (x_aug - self.per_channel_mean) / self.per_channel_std

        return self.model(x_aug_normalized)

    @staticmethod
    def normalized_index(a, b):
        return (a - b) / (a + b + EPS)


model = SKeMaBathyModel()

sample_input = torch.rand((2, 7, 512, 512), device=torch.device("cpu"), requires_grad=False)
model(sample_input)

In [ ]:
ckpt = torch.load("./Unet_tu-maxvit_tiny_tf_512_20250714_222203.ckpt", map_location="cpu")
state_dict = ckpt["state_dict"]

# Update keys
del state_dict["mean"]
del state_dict["std"]
model.load_state_dict(state_dict, strict=False)
model.eval()

In [ ]:
torch.onnx.export(
    model,
    sample_input,
    "./Unet_tu-maxvit_tiny_tf_512_20250714_222203.onnx",
    input_names=["input"],
    output_names=["output"],
    export_params=True,
    external_data=False,  # Store model weights in the model file
    opset_version=15,  # ONNX opset version
    do_constant_folding=True,  # Optimize constants
    verbose=False,
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
    # dynamic_shapes={"x": (torch.export.Dim("batch"), 5, 512, 512)},
    dynamo=False,
)